# Hindcast 0008-01: AO Tests

拆出 AO 与 O3 RMSE/EPFlux 的同步、异步和任意窗口 lag 扫描。基础 RMSE/EPFlux 表只用于支撑 AO 分析，不在本 notebook 额外画核心 RMSE 图。

In [ ]:
from __future__ import annotations

import glob
import math
import os
import re
import warnings
from pathlib import Path
from typing import Dict, Iterable, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter
from scipy.stats import linregress, pearsonr

plt.rcParams.update({
    "figure.facecolor": "white",
    "savefig.facecolor": "white",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.grid": True,
    "grid.linestyle": "--",
    "grid.alpha": 0.35,
})

CASE = "0008-01"
REF_YEAR = 8

REPO_ROOT = Path("/home/weiji/restart_exam/code_cleaned")
TEST_ROOT = REPO_ROOT / "Hindcast_experiment" / "TEST_TROPOS"
OUT_ROOT = TEST_ROOT / "outputs" / CASE
FIG_DIR = OUT_ROOT / "figures"
TABLE_DIR = OUT_ROOT / "tables"
CACHE_DIR = OUT_ROOT / "cache"
for d in [FIG_DIR, TABLE_DIR, CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DATA_ROOT = Path("/mnt/soclim0/public_data/weiji")
HINDCAST_ROOT = DATA_ROOT / "Hindcast"
BWCN_ROOT = DATA_ROOT / "BWCN"
B2000_ROOT = DATA_ROOT / "B2000WCN001002_timefixed"

CASE_ROOT = HINDCAST_ROOT / CASE

# Main windows. The EP window matches the old Hindcast_vertical_analysis scatter.
O3_END = (5, 30)
EP_WINDOW = ((1, 20), (2, 10))
EP_WINDOW_LABEL = "Jan20-Feb10"
AO_BASE_WINDOW = EP_WINDOW
AO_BASE_WINDOW_LABEL = EP_WINDOW_LABEL
AO_TEST_WINDOWS = {
    "Jan20-Feb10": EP_WINDOW,
    "Jan": ((1, 1), (1, 31)),
    "Jan-Feb": ((1, 1), (2, 28)),
    "Jan-Mar": ((1, 1), (3, 31)),
}
Z300_WINDOW = EP_WINDOW
Z300_WINDOW_LABEL = EP_WINDOW_LABEL
MONTH_WINDOWS = {
    "Jan": ((1, 1), (1, 31)),
    "Feb": ((2, 1), (2, 28)),
    "Mar": ((3, 1), (3, 31)),
    "Apr": ((4, 1), (4, 30)),
    "May": ((5, 1), (5, 30)),
}
MONTH_ORDER = ["Jan", "Feb", "Mar", "Apr", "May"]

LAT_EP = (40.0, 80.0)
LAT_POLAR = (60.0, 90.0)
LAT_Z300 = (20.0, 90.0)
PLEV_EP_PA = 10000.0
PLEV_EP_HPA = PLEV_EP_PA / 100.0
PLEV_U_PA = 1000.0
PLEV_Z300_PA = 30000.0
PLEV_Z300_HPA = PLEV_Z300_PA / 100.0

EP_SCALAR_DESCRIPTION = (
    f"mean -ep2 vertical EP-flux component at {PLEV_EP_HPA:.0f} hPa, "
    f"cos-lat mean {LAT_EP[0]:.0f}-{LAT_EP[1]:.0f}N, {EP_WINDOW_LABEL}; not EP-flux divergence"
)
Z300_PATTERN_DESCRIPTION = (
    f"{PLEV_Z300_HPA:.0f} hPa height, {Z300_WINDOW_LABEL} mean, "
    f"{LAT_Z300[0]:.0f}-{LAT_Z300[1]:.0f}N, zonal mean removed before pattern metrics"
)

WAVES = ["all_waves", "wave1", "wave2", "wave_rest"]
WAVE_LABELS = {
    "all_waves": "All waves",
    "wave1": "Wave 1",
    "wave2": "Wave 2",
    "wave_rest": "Wave rest",
    "wave1_plus_wave2": "Wave 1 + Wave 2",
}

# Expensive sections cache their results. Keep True for the full requested test.
RUN_Z300_DIAGNOSTICS = True
BUILD_U60N10_IF_MISSING = True
CLIM_MAX_B2000_YEARS_FOR_Z300 = None  # None = all B2000WCN001002 Z3 years available.

print(f"Output root: {OUT_ROOT}")
print(f"Case root exists: {CASE_ROOT.exists()} -> {CASE_ROOT}")

In [ ]:
# -----------------------------
# Shared helpers
# -----------------------------

def member_short_id(member) -> str:
    text = str(member)
    m = re.search(r"\.(\d{3})\.cam\.h3", text)
    if m:
        return m.group(1)
    m = re.search(r"\.(\d{3})\.", text)
    return m.group(1) if m else text


def date_parts(date_values):
    arr = np.asarray(date_values, dtype=np.int64)
    year = arr // 10000
    mmdd = arr % 10000
    month = mmdd // 100
    day = mmdd % 100
    return year, month, day


def date_mask(date_values, start=(1, 1), end=(5, 30), year: Optional[int] = None):
    yy, mm, dd = date_parts(date_values)
    start_key = start[0] * 100 + start[1]
    end_key = end[0] * 100 + end[1]
    key = mm * 100 + dd
    mask = (key >= start_key) & (key <= end_key)
    if year is not None:
        mask = mask & (yy == int(year))
    return mask


def one_dim_date(ds_or_da) -> np.ndarray:
    date = ds_or_da["date"]
    if "member" in date.dims:
        date = date.isel(member=0)
    return np.asarray(date.values, dtype=np.int64)


def assign_member_short_coord(da: xr.DataArray) -> xr.DataArray:
    if "member" not in da.dims:
        return da
    mids = [member_short_id(v) for v in da["member"].values]
    return da.assign_coords(member_short=("member", mids))


def select_latband(da: xr.DataArray, lat_range: Tuple[float, float], lat_name="lat") -> xr.DataArray:
    lat = da[lat_name]
    descending = float(lat.values[0]) > float(lat.values[-1])
    lo, hi = lat_range
    return da.sel({lat_name: slice(hi, lo) if descending else slice(lo, hi)})


def coslat_mean(da: xr.DataArray, lat_range: Optional[Tuple[float, float]] = None, lat_name="lat") -> xr.DataArray:
    if lat_range is not None:
        da = select_latband(da, lat_range, lat_name=lat_name)
    weights = np.cos(np.deg2rad(da[lat_name])).clip(0, 1)
    return da.weighted(weights.fillna(0)).mean(lat_name, skipna=True)


def finite_corr(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 3:
        return np.nan, np.nan
    r, p = pearsonr(x[mask], y[mask])
    return float(r), float(p)


def scatter_fit(ax, df, xcol, ycol, title, xlabel=None, ylabel=None, color="tab:blue", annotate_members=True):
    sub = df[[xcol, ycol, "member"]].replace([np.inf, -np.inf], np.nan).dropna()
    ax.scatter(sub[xcol], sub[ycol], s=62, color=color, edgecolor="black", linewidth=0.5, alpha=0.88)
    if annotate_members:
        for _, row in sub.iterrows():
            ax.text(row[xcol], row[ycol], str(row["member"]), fontsize=7, alpha=0.65)
    if len(sub) >= 3:
        fit = linregress(sub[xcol].values, sub[ycol].values)
        xx = np.linspace(sub[xcol].min(), sub[xcol].max(), 100)
        ax.plot(xx, fit.slope * xx + fit.intercept, color="crimson", ls="--", lw=1.8)
        ax.text(
            0.03, 0.97,
            f"R = {fit.rvalue:.3f}\nP = {fit.pvalue:.2e}",
            transform=ax.transAxes,
            va="top",
            ha="left",
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.84, edgecolor="0.7"),
        )
    ax.set_title(title)
    ax.set_xlabel(xlabel or xcol)
    ax.set_ylabel(ylabel or ycol)
    return ax


def savefig(fig, name):
    png = FIG_DIR / f"{name}.png"
    pdf = FIG_DIR / f"{name}.pdf"
    fig.savefig(png, dpi=300, bbox_inches="tight")
    fig.savefig(pdf, bbox_inches="tight")
    print(f"Saved: {png}")
    return png, pdf


def mmdd_label(date_values):
    _, mm, dd = date_parts(date_values)
    return np.array([f"{int(m):02d}-{int(d):02d}" for m, d in zip(mm, dd)])


def month_ticks(date_values):
    _, mm, dd = date_parts(date_values)
    positions, labels = [], []
    names = {1:"Jan", 2:"Feb", 3:"Mar", 4:"Apr", 5:"May", 6:"Jun"}
    seen = set()
    for i, (m, d) in enumerate(zip(mm, dd)):
        if int(d) == 1 and int(m) not in seen:
            positions.append(i)
            labels.append(names.get(int(m), str(int(m))))
            seen.add(int(m))
    return positions, labels


def standardize_1d(y):
    y = np.asarray(y, dtype=float)
    return (y - np.nanmean(y)) / np.nanstd(y)

In [ ]:
# -----------------------------
# Data loaders for the cleaned Hindcast products
# -----------------------------

def load_hindcast_o3(case=CASE, pressure_tag="30_70hPa"):
    path = HINDCAST_ROOT / case / "partial_O3" / f"{case}_partial_O3_all_ranges_members.nc"
    if not path.exists():
        raise FileNotFoundError(f"Missing cleaned partial O3 product: {path}")
    with xr.open_dataset(path, decode_times=False) as ds:
        var = f"O3_partial_60_90N_{pressure_tag}"
        da = assign_member_short_coord(ds[var]).load()
        date = one_dim_date(ds)
    mask = date_mask(date, start=(1, 1), end=O3_END)
    da = da.isel(lead_time=mask)
    date = date[mask]
    da = da.assign_coords(date=("lead_time", date))
    return da, date


def load_bwcn_ref_o3(year=REF_YEAR, pressure_tag="30_70hPa"):
    path = BWCN_ROOT / "partial_O3" / "BWCN_partial_O3_all_ranges.nc"
    if not path.exists():
        raise FileNotFoundError(path)
    with xr.open_dataset(path, decode_times=False) as ds:
        var = f"O3_partial_60_90N_{pressure_tag}"
        date = np.asarray(ds["date"].values, dtype=np.int64)
        mask = date_mask(date, start=(1, 1), end=O3_END, year=year)
        da = ds[var].isel(time=mask).load()
        date = date[mask]
    da = da.rename({"time": "lead_time"}).assign_coords(lead_time=np.arange(len(date)), date=("lead_time", date))
    return da, date


def load_epflux_wave(case=CASE, wave="all_waves", plev_pa=PLEV_EP_PA, lat_range=LAT_EP):
    path = HINDCAST_ROOT / case / "EPflux_daily_ubar" / wave / f"EPFLUX_{wave}_{case}_members_time_plev_lat.nc"
    if not path.exists():
        raise FileNotFoundError(path)
    with xr.open_dataset(path, decode_times=False) as ds:
        ep2 = assign_member_short_coord(ds["ep2"])
        ep2_100 = ep2.sel(plev=plev_pa, method="nearest")
        ep2_100 = coslat_mean(ep2_100, lat_range=lat_range)
        # Use -ep2 so positive means upward wave activity, matching the old Fz scatter convention.
        out = (-ep2_100).load()
    return out


def ep_window_mean(ep_da: xr.DataArray, date_values, start_end=EP_WINDOW):
    start, end = start_end
    mask = date_mask(date_values, start=start, end=end)
    return ep_da.isel(lead_time=mask).mean("lead_time", skipna=True)


def load_all_wave_metrics(case=CASE, date_values=None, start_end=EP_WINDOW):
    if date_values is None:
        _, date_values = load_hindcast_o3(case)
    rows = []
    series = {}
    for wave in WAVES:
        da = load_epflux_wave(case, wave=wave)
        da = da.isel(lead_time=slice(0, len(date_values)))
        metric = ep_window_mean(da, date_values, start_end=start_end)
        series[wave] = da.assign_coords(date=("lead_time", date_values[: da.sizes["lead_time"]]))
        for member, val in zip(metric["member_short"].values, metric.values):
            rows.append({"member": str(member), "wave": wave, "ep100_upward": float(val)})
    return pd.DataFrame(rows), series


def compute_rmse_table(o3_hind: xr.DataArray, date_h, o3_ref: xr.DataArray, date_ref, start=(1, 1), end=O3_END, label="Jan01-May30"):
    mh = date_mask(date_h, start=start, end=end)
    mr = date_mask(date_ref, start=start, end=end, year=REF_YEAR)
    h = o3_hind.isel(lead_time=mh)
    r = o3_ref.isel(lead_time=mr)
    n = min(h.sizes["lead_time"], r.sizes["lead_time"])
    h = h.isel(lead_time=slice(0, n))
    r = r.isel(lead_time=slice(0, n))
    diff = h - r
    rmse = np.sqrt((diff ** 2).mean("lead_time", skipna=True))
    return pd.DataFrame({
        "member": [str(v) for v in rmse["member_short"].values],
        "RMSE_DU": rmse.values.astype(float),
        "rmse_window": label,
        "n_days": n,
    }).sort_values("RMSE_DU").reset_index(drop=True)


def merge_rmse_ep(rmse_df, ep_metric_df):
    wide = ep_metric_df.pivot(index="member", columns="wave", values="ep100_upward").reset_index()
    wide = wide.rename(columns={w: f"EP100_{w}" for w in WAVES})
    return rmse_df.merge(wide, on="member", how="left")

In [ ]:
# -----------------------------
# Product sanity check for 0008-01
# -----------------------------
expected = {
    "partial_O3": CASE_ROOT / "partial_O3" / f"{CASE}_partial_O3_all_ranges_members.nc",
    "EP all": CASE_ROOT / "EPflux_daily_ubar" / "all_waves" / f"EPFLUX_all_waves_{CASE}_members_time_plev_lat.nc",
    "EP wave1": CASE_ROOT / "EPflux_daily_ubar" / "wave1" / f"EPFLUX_wave1_{CASE}_members_time_plev_lat.nc",
    "EP wave2": CASE_ROOT / "EPflux_daily_ubar" / "wave2" / f"EPFLUX_wave2_{CASE}_members_time_plev_lat.nc",
    "EP rest": CASE_ROOT / "EPflux_daily_ubar" / "wave_rest" / f"EPFLUX_wave_rest_{CASE}_members_time_plev_lat.nc",
    "FWD": CASE_ROOT / "final_warming_date" / f"{CASE}_FWD_plev_member.nc",
    "AO/NAM projection": CASE_ROOT / "NAM_B2000WCN_projection" / f"{CASE}_AO_NAM_B2000WCN_projection_members.nc",
}
for name, path in expected.items():
    print(f"{name:18s}: {path.exists()}  {path}")

with xr.open_dataset(expected["partial_O3"], decode_times=False) as ds:
    print("\npartial_O3 dims:", dict(ds.sizes))
    print("partial_O3 vars:", list(ds.data_vars))
with xr.open_dataset(expected["EP all"], decode_times=False) as ds:
    print("\nEP all dims:", dict(ds.sizes))
    print("EP attrs:", {k: ds.attrs.get(k) for k in ["method", "do_ubar", "use_omega_w_correction"]})

In [ ]:
# -----------------------------
# Figure routing for the split notebook
# -----------------------------
NOTEBOOK_STEM = "Hindcast_0008_01_02_ao_tests"

def use_figure_dir(content_tag: str):
    global FIG_DIR
    FIG_DIR = OUT_ROOT / "figures" / NOTEBOOK_STEM / content_tag
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    print("Figure dir:", FIG_DIR)
    return FIG_DIR


In [ ]:
# -----------------------------
# Shared scalar data prep for split diagnostics
# -----------------------------
o3_h, date_h = load_hindcast_o3(CASE)
o3_ref, date_ref = load_bwcn_ref_o3(REF_YEAR)
ep_metric_df, ep_series = load_all_wave_metrics(CASE, date_h, EP_WINDOW)
rmse_all = compute_rmse_table(o3_h, date_h, o3_ref, date_ref, start=(1, 1), end=O3_END, label="Jan01-May30")
rmse_ep_all = merge_rmse_ep(rmse_all, ep_metric_df)
rmse_ep_all["EP100_wave1_plus_wave2"] = rmse_ep_all["EP100_wave1"] + rmse_ep_all["EP100_wave2"]
EP_SCALAR_COLS = [
    ("EP100_all_waves", "all_waves", "All", "black"),
    ("EP100_wave1", "wave1", "W1", "tab:blue"),
    ("EP100_wave2", "wave2", "W2", "tab:orange"),
    ("EP100_wave_rest", "wave_rest", "Rest", "tab:green"),
    ("EP100_wave1_plus_wave2", "wave1_plus_wave2", "W1+W2", "tab:purple"),
]
print("Prepared base tables:", rmse_ep_all.shape)


## 图：AO 与 EPFlux/O3 RMSE

**做什么**：检查 B2000WCN 第一模态投影得到的 Hindcast AO 是否与 O3 RMSE 或同窗口 EPFlux 有关。

**怎么做**：AO 使用 cleaned `NAM_B2000WCN_projection`，即 1000 hPa zonal-mean AO 投影到 B2000WCN 第一模态。基础图使用 `Jan20-Feb10`，让 `AO vs O3 RMSE` 和 `AO vs concurrent all-wave EP` 的 AO 时间窗口一致。随后扩展到 `Jan`、`Jan-Feb`、`Jan-Mar`；每个扩展窗口都会重新计算同窗口的 EP100 分波指标。

**科学问题**：annular mode 状态是否能解释早期/累积波活动异常，进而解释后续臭氧可预报性差异？

**预期**：如果 AO 是波活动异常的低维源头，AO 与同窗口 EP100 或 O3 RMSE 应有系统相关，并可能随窗口扩展增强或减弱。

**运行后解读**：待图生成后填写。


In [ ]:
use_figure_dir("01_concurrent_ao_rmse_epflux")


In [ ]:
# -----------------------------
# AO test: use B2000WCN-projected AO/NAM if available.
# -----------------------------

def load_projected_ao(case=CASE):
    new_path = HINDCAST_ROOT / case / "NAM_B2000WCN_projection" / f"{case}_AO_NAM_B2000WCN_projection_members.nc"
    if not new_path.exists():
        print(f"Missing projected AO/NAM product: {new_path}")
        print("The legacy NAM/*.nc file lacks a reliable member axis for this scatter, so AO tests are skipped until the projected product is generated.")
        return None, None
    with xr.open_dataset(new_path, decode_times=False) as ds:
        ao = assign_member_short_coord(ds["AO_Index"]).load()
        date = one_dim_date(ds)
    if int(ao.count()) == 0:
        print("Projected AO file exists, but AO_Index is all NaN. Regenerate it with the fixed script:")
        print("/home/weiji/miniconda3/envs/jimnew/bin/python Hindcast_experiment/date_treatment/scripts/compute_hindcast_ao_nam_b2000wcn_projection.py --cases 0008-01 --overwrite --max-workers 4")
        return None, None
    ao = ao.assign_coords(date=("lead_time", date))
    return ao, date


def window_tag(label: str) -> str:
    return label.replace("+", "plus").replace("-", "_").replace(" ", "_")


def ao_mean_for_window(ao_da: xr.DataArray, date_values: np.ndarray, start_end) -> xr.DataArray:
    mask = date_mask(date_values, start=start_end[0], end=start_end[1])
    return ao_da.isel(lead_time=mask).mean("lead_time", skipna=True)


def ep_metrics_for_window(ep_series_dict: Dict[str, xr.DataArray], date_values: np.ndarray, start_end) -> pd.DataFrame:
    rows = []
    for wave, da in ep_series_dict.items():
        metric = ep_window_mean(da, date_values, start_end=start_end)
        for member, val in zip(metric["member_short"].values, metric.values):
            rows.append({"member": str(member), "wave": wave, "ep100_upward": float(val)})
    wide = pd.DataFrame(rows).pivot(index="member", columns="wave", values="ep100_upward").reset_index()
    wide = wide.rename(columns={w: f"EP100_{w}" for w in WAVES})
    wide["EP100_wave1_plus_wave2"] = wide["EP100_wave1"] + wide["EP100_wave2"]
    return wide


def build_ao_ep_window_table(ao_da: xr.DataArray, date_values: np.ndarray, label: str, start_end) -> pd.DataFrame:
    ao_metric = ao_mean_for_window(ao_da, date_values, start_end)
    rows = [
        {"member": str(mid), "AO_mean": float(ao_metric.isel(member=i)), "AO_window": label}
        for i, mid in enumerate(ao_da["member_short"].values)
    ]
    ao_df = pd.DataFrame(rows)
    ep_df = ep_metrics_for_window(ep_series, date_h, start_end)
    out = rmse_all.merge(ao_df, on="member", how="left").merge(ep_df, on="member", how="left")
    return out


ao, date_ao = load_projected_ao(CASE)
if ao is not None:
    ao_window_tables = []
    for label, start_end in AO_TEST_WINDOWS.items():
        df = build_ao_ep_window_table(ao, date_ao, label, start_end)
        ao_window_tables.append(df)
    ao_all_windows = pd.concat(ao_window_tables, ignore_index=True)
    ao_all_windows.to_csv(TABLE_DIR / f"{CASE}_AO_EP_RMSE_metrics_by_window.csv", index=False)

    # Backward-compatible base table/figures now use Jan20-Feb10 AO for every AO panel.
    ao_join = ao_all_windows.loc[ao_all_windows["AO_window"] == AO_BASE_WINDOW_LABEL].copy()
    ao_join.to_csv(TABLE_DIR / f"{CASE}_AO_EP_RMSE_metrics.csv", index=False)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), constrained_layout=True)
    scatter_fit(
        axes[0], ao_join, "AO_mean", "RMSE_DU",
        "AO vs O3 RMSE", f"AO mean {AO_BASE_WINDOW_LABEL}", "O3 RMSE (DU)", "tab:cyan",
    )
    scatter_fit(
        axes[1], ao_join, "AO_mean", "EP100_all_waves",
        "AO vs concurrent all-wave EP", f"AO mean {AO_BASE_WINDOW_LABEL}",
        f"100 hPa all-wave -EP2 ({AO_BASE_WINDOW_LABEL})", "tab:cyan",
    )
    fig.suptitle(f"AO diagnostics, {AO_BASE_WINDOW_LABEL}", fontsize=12)
    savefig(fig, f"{CASE}_AO_tests_RMSE_EPFlux")
    plt.show()

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), constrained_layout=True)
    for ax, label in zip(axes, ["Jan", "Jan-Feb"]):
        df = ao_all_windows.loc[ao_all_windows["AO_window"] == label]
        scatter_fit(
            ax, df, "AO_mean", "RMSE_DU",
            f"{label}: AO vs O3 RMSE", f"AO mean {label}", "O3 RMSE (DU)", "tab:cyan",
        )
    savefig(fig, f"{CASE}_AO_vs_RMSE_Jan_JanFeb")
    plt.show()

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), constrained_layout=True)
    for ax, label in zip(axes, ["Jan-Mar", "Jan20-Feb10"]):
        df = ao_all_windows.loc[ao_all_windows["AO_window"] == label]
        scatter_fit(
            ax, df, "AO_mean", "RMSE_DU",
            f"{label}: AO vs O3 RMSE", f"AO mean {label}", "O3 RMSE (DU)", "tab:cyan",
        )
    savefig(fig, f"{CASE}_AO_vs_RMSE_JanMar_Jan20Feb10")
    plt.show()

    fig, axes = plt.subplots(len(AO_TEST_WINDOWS), 2, figsize=(12.5, 14.2), constrained_layout=True)
    for r, (label, _) in enumerate(AO_TEST_WINDOWS.items()):
        df = ao_all_windows.loc[ao_all_windows["AO_window"] == label]
        scatter_fit(
            axes[r, 0], df, "AO_mean", "RMSE_DU",
            f"{label}: AO vs O3 RMSE", f"AO mean {label}", "O3 RMSE (DU)", "tab:cyan",
            annotate_members=False,
        )
        scatter_fit(
            axes[r, 1], df, "AO_mean", "EP100_all_waves",
            f"{label}: AO vs all-wave EP", f"AO mean {label}", f"EP100 all ({label})", "tab:cyan",
            annotate_members=False,
        )
    fig.suptitle("AO window sensitivity: AO, O3 RMSE, and concurrent all-wave EP100", fontsize=12)
    savefig(fig, f"{CASE}_AO_tests_RMSE_EPFlux_windows")
    plt.show()

    fig, axes = plt.subplots(2, 2, figsize=(12, 9), constrained_layout=True)
    for ax, wave in zip(axes.ravel(), WAVES):
        scatter_fit(
            ax, ao_join, "AO_mean", f"EP100_{wave}",
            f"AO vs {WAVE_LABELS[wave]}", f"AO mean {AO_BASE_WINDOW_LABEL}",
            f"{WAVE_LABELS[wave]} -EP2 ({AO_BASE_WINDOW_LABEL})", "tab:cyan", annotate_members=False,
        )
    fig.suptitle(f"AO vs EP100 wave components, {AO_BASE_WINDOW_LABEL}", fontsize=12)
    savefig(fig, f"{CASE}_AO_vs_wave_components")
    plt.show()

    wave_cols = [
        ("EP100_all_waves", "All"),
        ("EP100_wave1", "W1"),
        ("EP100_wave2", "W2"),
        ("EP100_wave_rest", "Rest"),
        ("EP100_wave1_plus_wave2", "W1+W2"),
    ]
    fig, axes = plt.subplots(len(AO_TEST_WINDOWS), len(wave_cols), figsize=(18.5, 14.2), constrained_layout=True)
    for r, (label, _) in enumerate(AO_TEST_WINDOWS.items()):
        df = ao_all_windows.loc[ao_all_windows["AO_window"] == label]
        for c, (col, short_label) in enumerate(wave_cols):
            scatter_fit(
                axes[r, c], df, "AO_mean", col,
                f"{label}: AO vs {short_label}", f"AO mean {label}", f"EP100 {short_label}",
                "tab:cyan", annotate_members=False,
            )
    fig.suptitle("AO window sensitivity: concurrent EP100 wave components", fontsize=12)
    savefig(fig, f"{CASE}_AO_vs_wave_components_windows")
    plt.show()

    # Also split the 4 x 5 window-sensitivity panel into three 2 x 2 figures.
    # Each figure shows the four AO windows: Jan20-Feb10, Jan, Jan-Feb, and Jan-Mar.
    def plot_ao_metric_windows_2x2(metric_col: str, metric_label: str, out_suffix: str):
        fig, axes = plt.subplots(2, 2, figsize=(11.5, 9.0), constrained_layout=True)
        for ax, (label, _) in zip(axes.ravel(), AO_TEST_WINDOWS.items()):
            df = ao_all_windows.loc[ao_all_windows["AO_window"] == label]
            scatter_fit(
                ax, df, "AO_mean", metric_col,
                f"{label}: AO vs {metric_label}", f"AO mean {label}", f"EP100 {metric_label} ({label})",
                "tab:cyan", annotate_members=False,
            )
        fig.suptitle(f"AO window sensitivity: AO vs EP100 {metric_label}", fontsize=12)
        savefig(fig, f"{CASE}_AO_vs_{out_suffix}_windows")
        plt.show()

    plot_ao_metric_windows_2x2("EP100_all_waves", "All waves", "all_waves")
    plot_ao_metric_windows_2x2("EP100_wave1_plus_wave2", "W1+W2", "wave1_plus_wave2")
    plot_ao_metric_windows_2x2("EP100_wave_rest", "Rest", "wave_rest")


## 图：AO 领先、EPFlux 滞后的异步窗口测试

**做什么**：测试 AO 是否先于进入平流层的 wave activity。固定测试采用 30 天 AO 窗口和 30 天 EPFlux 窗口，EPFlux 窗口相对 AO 晚 10 天；同时扫描不同 AO 起始日、window 长度和 delay，寻找 member-to-member 相关最强的时间组合。

**怎么做**：AO 指标是 B2000WCN-projected AO 的成员平均值；EPFlux 指标仍为 `EP100 = mean(-ep2)`，即 100 hPa、40-80N cos-lat 平均的垂直 EP-flux 分量，不是 divergence。固定测试锚定原先的 Jan20 EPFlux 起始日，因此 AO = Jan10-Feb08，EPFlux = Jan20-Feb18。扫描只考虑 EPFlux 晚于 AO 的正 delay。

**科学问题**：如果早期 AO 调制后续 planetary-wave 入射，那么 AO 窗口应当与稍晚的 all-wave、wave1+wave2 或 rest EP100 有显著相关；若只有同步相关而无滞后相关，AO 更可能只是同期环流状态的伴随指标。

**预期**：优先关注 `|R| >= 0.5` 且 `p < 0.05` 的组合；若固定 10 天 lag 或扫描最佳组合都不显著，就不把 AO lead 作为主要 source 解释。

**运行后解读**：当前 0008-01 运行结果显示，固定 30 天窗口、AO 领先 10 天的测试不显著：all-wave `R≈0.20, p≈0.30`，W1+W2 `R≈0.15, p≈0.42`。探索扫描里显著点主要集中在 20 天窗口，尤其早 1 月 AO vs 稍晚 EPFlux 的 wave2/W1+W2 负相关和 rest 正相关；因此额外输出 20 天 heatmap、20 天窗口/5 天 lag 的相关性折线图，以及 AO Jan01-Jan20 vs EP Jan06-Jan25 的 all-wave/W1+W2 散点图。指定窗口读数为：All `R=-0.622, p=2.45e-4`，W1+W2 `R=-0.699, p=1.7e-5`，wave2 `R=-0.761, p=1.04e-6`，rest `R=0.693, p=2.2e-5`。若只坚持 30 天、10 天 lag，目前不应把 AO lead 作为强 source 结论。

In [ ]:
use_figure_dir("02_ao_lead_epflux_lag_scan")


In [ ]:
# -----------------------------
# AO lead / delayed EPFlux tests
# -----------------------------
# Fixed hypothesis: AO is averaged over a 30-day window, EPFlux is averaged over
# the following 30-day window delayed by +10 days. The fixed test anchors the
# EPFlux start date at Jan20, so AO = Jan10-Feb08 and EPFlux = Jan20-Feb18.
# The scan then varies AO start date, window length, and delay to see whether a
# stronger, physically coherent lag emerges.

if "date_h" not in globals() or "ep_series" not in globals():
    o3_h, date_h = load_hindcast_o3(CASE)
    _, ep_series = load_all_wave_metrics(CASE, date_h, EP_WINDOW)

if "load_projected_ao" not in globals():
    def load_projected_ao(case=CASE):
        new_path = HINDCAST_ROOT / case / "NAM_B2000WCN_projection" / f"{case}_AO_NAM_B2000WCN_projection_members.nc"
        if not new_path.exists():
            print(f"Missing projected AO/NAM product: {new_path}")
            return None, None
        with xr.open_dataset(new_path, decode_times=False) as ds:
            ao = assign_member_short_coord(ds["AO_Index"]).load()
            date = one_dim_date(ds)
        if int(ao.count()) == 0:
            print("Projected AO file exists, but AO_Index is all NaN. Regenerate Hindcast AO/NAM first.")
            return None, None
        return ao.assign_coords(date=("lead_time", date)), date

if "ao" not in globals() or "date_ao" not in globals():
    ao, date_ao = load_projected_ao(CASE)

_DOY_MONTH_START = np.array([0, 31, 59, 90, 120, 151, 181, 212, 243, 273, 304, 334], dtype=int)
_MONTH_ABBR = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]


def mmdd_to_doy(month: int, day: int) -> int:
    return int(_DOY_MONTH_START[int(month) - 1] + int(day))


def date_to_doy(date_values) -> np.ndarray:
    _, month, day = date_parts(date_values)
    return _DOY_MONTH_START[month.astype(int) - 1] + day.astype(int)


def doy_to_mmdd(doy: int) -> Tuple[int, int]:
    doy = int(doy)
    month = int(np.searchsorted(_DOY_MONTH_START + 1, doy, side="right"))
    month = min(max(month, 1), 12)
    day = doy - int(_DOY_MONTH_START[month - 1])
    return month, day


def doy_window_label(start_doy: int, window_days: int) -> str:
    sm, sd = doy_to_mmdd(start_doy)
    em, ed = doy_to_mmdd(start_doy + window_days - 1)
    return f"{_MONTH_ABBR[sm - 1]}{sd:02d}-{_MONTH_ABBR[em - 1]}{ed:02d}"


def doy_window_mask(date_values, start_doy: int, window_days: int) -> np.ndarray:
    doy = date_to_doy(date_values)
    end_doy = int(start_doy) + int(window_days) - 1
    return (doy >= int(start_doy)) & (doy <= end_doy)


def member_mean_for_doy_window(da: xr.DataArray, date_values, start_doy: int, window_days: int) -> xr.DataArray:
    n = min(int(da.sizes["lead_time"]), len(date_values))
    mask = doy_window_mask(np.asarray(date_values)[:n], start_doy, window_days)
    if mask.sum() == 0:
        return da.isel(lead_time=slice(0, 0)).mean("lead_time", skipna=True)
    return da.isel(lead_time=np.where(mask)[0]).mean("lead_time", skipna=True)


def ao_ep_async_table(
    ao_da: xr.DataArray,
    ao_date_values,
    ep_series_dict: Dict[str, xr.DataArray],
    ep_date_values,
    ao_start_doy: int,
    window_days: int,
    delay_days: int,
) -> pd.DataFrame:
    ep_start_doy = int(ao_start_doy) + int(delay_days)
    ao_metric = member_mean_for_doy_window(ao_da, ao_date_values, ao_start_doy, window_days)
    ao_df = pd.DataFrame({
        "member": [str(mid) for mid in ao_metric["member_short"].values],
        "AO_mean": ao_metric.values.astype(float),
    })

    ep_wide = None
    for wave in WAVES:
        metric = member_mean_for_doy_window(ep_series_dict[wave], ep_date_values, ep_start_doy, window_days)
        this = pd.DataFrame({
            "member": [str(mid) for mid in metric["member_short"].values],
            f"EP100_{wave}": metric.values.astype(float),
        })
        ep_wide = this if ep_wide is None else ep_wide.merge(this, on="member", how="outer")
    ep_wide["EP100_wave1_plus_wave2"] = ep_wide["EP100_wave1"] + ep_wide["EP100_wave2"]

    out = ao_df.merge(ep_wide, on="member", how="inner")
    out["ao_start_doy"] = int(ao_start_doy)
    out["ep_start_doy"] = int(ep_start_doy)
    out["window_days"] = int(window_days)
    out["delay_days"] = int(delay_days)
    out["AO_window"] = doy_window_label(ao_start_doy, window_days)
    out["EP_window"] = doy_window_label(ep_start_doy, window_days)
    return out


ASYNC_WAVE_COLS = [
    ("EP100_all_waves", "All waves", "black"),
    ("EP100_wave1", "Wave 1", "tab:blue"),
    ("EP100_wave2", "Wave 2", "tab:orange"),
    ("EP100_wave_rest", "Rest", "tab:green"),
    ("EP100_wave1_plus_wave2", "W1+W2", "tab:purple"),
]


def ao_ep_corr_rows(df: pd.DataFrame) -> list[dict]:
    rows = []
    for col, label, _ in ASYNC_WAVE_COLS:
        r, p = finite_corr(df["AO_mean"], df[col])
        rows.append({
            "metric": col.replace("EP100_", ""),
            "label": label,
            "R_AO_vs_EP100": r,
            "P": p,
            "ao_start_doy": int(df["ao_start_doy"].iloc[0]),
            "ep_start_doy": int(df["ep_start_doy"].iloc[0]),
            "window_days": int(df["window_days"].iloc[0]),
            "delay_days": int(df["delay_days"].iloc[0]),
            "AO_window": str(df["AO_window"].iloc[0]),
            "EP_window": str(df["EP_window"].iloc[0]),
            "n_members": int(df[["AO_mean", col]].dropna().shape[0]),
            "definition": "AO mean leads EP100; EP100 is mean(-ep2), 100 hPa, 40-80N, not divergence.",
        })
    return rows


if ao is None:
    print("Skip asynchronous AO/EPFlux test: projected AO is unavailable.")
else:
    ASYNC_WINDOW_DAYS = 30
    ASYNC_DELAY_DAYS = 10
    ASYNC_EP_ANCHOR_START = (1, 20)
    ASYNC_EP_ANCHOR_DOY = mmdd_to_doy(*ASYNC_EP_ANCHOR_START)
    ASYNC_AO_START_DOY = ASYNC_EP_ANCHOR_DOY - ASYNC_DELAY_DAYS

    fixed_async = ao_ep_async_table(
        ao, date_ao, ep_series, date_h,
        ao_start_doy=ASYNC_AO_START_DOY,
        window_days=ASYNC_WINDOW_DAYS,
        delay_days=ASYNC_DELAY_DAYS,
    )
    fixed_async.to_csv(TABLE_DIR / f"{CASE}_AO_leads_EPFlux_lag10_30day_metrics.csv", index=False)
    fixed_corr = pd.DataFrame(ao_ep_corr_rows(fixed_async))
    fixed_corr.to_csv(TABLE_DIR / f"{CASE}_AO_leads_EPFlux_lag10_30day_correlations.csv", index=False)
    print("Fixed asynchronous test: AO leads EPFlux by 10 days, 30-day windows")
    print(fixed_corr[["label", "R_AO_vs_EP100", "P", "AO_window", "EP_window", "n_members"]])

    fig, axes = plt.subplots(2, 3, figsize=(15.6, 8.8), constrained_layout=True)
    for ax, (col, label, color) in zip(axes.ravel()[:5], ASYNC_WAVE_COLS):
        scatter_fit(
            ax, fixed_async, "AO_mean", col,
            f"AO lead 10d vs {label}",
            f"AO mean {fixed_async['AO_window'].iloc[0]}",
            f"EP100 {label} {fixed_async['EP_window'].iloc[0]}",
            color=color,
            annotate_members=False,
        )
    axes.ravel()[5].axis("off")
    axes.ravel()[5].text(
        0.0, 0.95,
        "Fixed hypothesis\n"
        f"AO: {fixed_async['AO_window'].iloc[0]}\n"
        f"EPFlux: {fixed_async['EP_window'].iloc[0]}\n"
        "Window: 30 d\nDelay: +10 d\n"
        "EP100 = mean(-ep2), 100 hPa, 40-80N\n"
        "Positive EP100 follows the upward-Fz convention.",
        va="top", ha="left", fontsize=10,
    )
    fig.suptitle("Asynchronous AO -> EPFlux test: AO leads wave activity by 10 days", fontsize=12)
    savefig(fig, f"{CASE}_AO_leads_EPFlux_lag10_30day")
    plt.show()

    scan_windows = [20, 30, 40]
    scan_delays = [0, 5, 10, 15, 20, 30, 40]
    scan_start_step = 5
    max_ep_doy = int(np.nanmax(date_to_doy(date_h)))
    max_ao_doy = int(np.nanmax(date_to_doy(date_ao)))
    scan_rows = []
    for window_days in scan_windows:
        for delay_days in scan_delays:
            max_start = min(max_ep_doy - delay_days - window_days + 1, max_ao_doy - window_days + 1)
            for ao_start_doy in range(1, max_start + 1, scan_start_step):
                df = ao_ep_async_table(
                    ao, date_ao, ep_series, date_h,
                    ao_start_doy=ao_start_doy,
                    window_days=window_days,
                    delay_days=delay_days,
                )
                scan_rows.extend(ao_ep_corr_rows(df))
    async_scan = pd.DataFrame(scan_rows)
    async_scan["abs_R"] = async_scan["R_AO_vs_EP100"].abs()
    async_scan["passes_nominal_filter"] = (async_scan["abs_R"] >= 0.5) & (async_scan["P"] < 0.05)
    async_scan = async_scan.sort_values(["passes_nominal_filter", "abs_R", "P"], ascending=[False, False, True])
    async_scan.to_csv(TABLE_DIR / f"{CASE}_AO_lead_EPFlux_lag_window_scan.csv", index=False)

    print("\nTop asynchronous AO -> EPFlux correlations from exploratory scan:")
    print(async_scan.head(15)[[
        "label", "R_AO_vs_EP100", "P", "AO_window", "EP_window", "window_days", "delay_days", "passes_nominal_filter"
    ]])

    def plot_async_heatmaps_for_window(window_days: int):
        scan_window = async_scan.loc[async_scan["window_days"] == window_days].copy()
        fig, axes = plt.subplots(2, 3, figsize=(16.5, 8.8), constrained_layout=True)
        heat_axes = axes.ravel()[:5]
        im = None
        for ax, (metric_col, label, _) in zip(heat_axes, ASYNC_WAVE_COLS):
            metric = metric_col.replace("EP100_", "")
            sub = scan_window.loc[scan_window["metric"] == metric].copy()
            pivot = sub.pivot(index="delay_days", columns="ao_start_doy", values="R_AO_vs_EP100").sort_index()
            im = ax.imshow(
                pivot.values,
                origin="lower",
                aspect="auto",
                cmap="RdBu_r",
                vmin=-1,
                vmax=1,
                extent=[pivot.columns.min(), pivot.columns.max(), pivot.index.min(), pivot.index.max()],
            )
            good = sub.loc[sub["passes_nominal_filter"]]
            if not good.empty:
                ax.scatter(good["ao_start_doy"], good["delay_days"], s=22, facecolors="none", edgecolors="black", linewidths=0.9)
            xticks = [mmdd_to_doy(1, 1), mmdd_to_doy(2, 1), mmdd_to_doy(3, 1), mmdd_to_doy(4, 1)]
            ax.set_xticks([t for t in xticks if pivot.columns.min() <= t <= pivot.columns.max()])
            ax.set_xticklabels([doy_window_label(int(t), 1)[:5] for t in ax.get_xticks()], rotation=30, ha="right")
            ax.set_yticks(scan_delays)
            ax.set_title(f"AO vs {label}")
            ax.set_xlabel("AO window start")
            ax.set_ylabel("EP lag after AO start (days)")
        axes.ravel()[5].axis("off")
        axes.ravel()[5].text(
            0.0, 0.95,
            "Exploratory scan\n"
            f"Shown: {window_days}-day windows\n"
            "Color: Pearson R across 30 members\n"
            "Open circles: |R| >= 0.5 and p < 0.05\n"
            "Only positive delays are tested, so EPFlux is concurrent with or later than AO.",
            va="top", ha="left", fontsize=10,
        )
        fig.colorbar(im, ax=heat_axes, shrink=0.88, label="R(AO, delayed EP100)")
        fig.suptitle(f"AO lead / EPFlux lag scan, {window_days}-day windows", fontsize=12)
        savefig(fig, f"{CASE}_AO_lead_EPFlux_lag_scan_{window_days}day_heatmaps")
        plt.show()

    plot_async_heatmaps_for_window(30)
    plot_async_heatmaps_for_window(20)

    line_window_days = 20
    line_delay_days = 5
    line_df = async_scan.loc[
        (async_scan["window_days"] == line_window_days)
        & (async_scan["delay_days"] == line_delay_days)
    ].copy()
    line_df.to_csv(TABLE_DIR / f"{CASE}_AO_EPFlux_20day_lag5_correlation_lines.csv", index=False)

    fig, ax = plt.subplots(figsize=(10.8, 5.8), constrained_layout=True)
    for metric_col, label, color in ASYNC_WAVE_COLS:
        metric = metric_col.replace("EP100_", "")
        sub = line_df.loc[line_df["metric"] == metric].sort_values("ao_start_doy")
        ax.plot(sub["ao_start_doy"], sub["R_AO_vs_EP100"], marker="o", ms=4.5, lw=1.8, color=color, label=label)
        sig = sub.loc[(sub["R_AO_vs_EP100"].abs() >= 0.5) & (sub["P"] < 0.05)]
        if not sig.empty:
            ax.scatter(sig["ao_start_doy"], sig["R_AO_vs_EP100"], s=64, facecolors="none", edgecolors=color, linewidths=1.4)
    xticks = [mmdd_to_doy(1, 1), mmdd_to_doy(1, 15), mmdd_to_doy(2, 1), mmdd_to_doy(2, 15), mmdd_to_doy(3, 1), mmdd_to_doy(3, 15), mmdd_to_doy(4, 1)]
    ax.set_xticks([t for t in xticks if line_df["ao_start_doy"].min() <= t <= line_df["ao_start_doy"].max()])
    ax.set_xticklabels([doy_window_label(int(t), 1)[:5] for t in ax.get_xticks()], rotation=30, ha="right")
    ax.axhline(0, color="0.25", lw=0.9)
    ax.axhline(0.5, color="0.45", lw=0.8, ls=":")
    ax.axhline(-0.5, color="0.45", lw=0.8, ls=":")
    ax.set_xlabel("AO 20-day window start")
    ax.set_ylabel("R(AO mean, delayed EP100)")
    ax.set_title("20-day windows with EPFlux lagged by 5 days")
    ax.legend(ncol=3, fontsize=9)
    savefig(fig, f"{CASE}_AO_EPFlux_20day_lag5_correlation_lines")
    plt.show()

    jan01_20 = ao_ep_async_table(
        ao, date_ao, ep_series, date_h,
        ao_start_doy=mmdd_to_doy(1, 1),
        window_days=20,
        delay_days=5,
    )
    jan01_20.to_csv(TABLE_DIR / f"{CASE}_AO_Jan01_Jan20_EP_Jan06_Jan25_all_w1plusw2.csv", index=False)
    fig, axes = plt.subplots(1, 2, figsize=(12.6, 5.2), constrained_layout=True)
    scatter_fit(
        axes[0], jan01_20, "AO_mean", "EP100_all_waves",
        "AO Jan01-Jan20 vs all-wave EP100 Jan06-Jan25",
        "AO mean Jan01-Jan20", "EP100 all waves Jan06-Jan25", "black",
        annotate_members=True,
    )
    scatter_fit(
        axes[1], jan01_20, "AO_mean", "EP100_wave1_plus_wave2",
        "AO Jan01-Jan20 vs W1+W2 EP100 Jan06-Jan25",
        "AO mean Jan01-Jan20", "EP100 W1+W2 Jan06-Jan25", "tab:purple",
        annotate_members=True,
    )
    fig.suptitle("AO lead test: 20-day window, 5-day lag; EP100 = mean(-ep2), 100 hPa, 40-80N", fontsize=11)
    savefig(fig, f"{CASE}_AO_Jan01_Jan20_vs_EP_Jan06_Jan25_all_w1plusw2")
    plt.show()

    robust = async_scan.loc[async_scan["passes_nominal_filter"]].copy()
    if robust.empty:
        print("No AO lead / delayed EPFlux combinations passed |R| >= 0.5 and p < 0.05; detailed scatter figure skipped.")
    else:
        top = robust.head(6)
        fig, axes = plt.subplots(2, 3, figsize=(16.0, 9.2), constrained_layout=True)
        color_lookup = {col: color for col, _, color in ASYNC_WAVE_COLS}
        label_lookup = {col: label for col, label, _ in ASYNC_WAVE_COLS}
        for ax, (_, row) in zip(axes.ravel(), top.iterrows()):
            col = f"EP100_{row['metric']}"
            df = ao_ep_async_table(
                ao, date_ao, ep_series, date_h,
                ao_start_doy=int(row["ao_start_doy"]),
                window_days=int(row["window_days"]),
                delay_days=int(row["delay_days"]),
            )
            scatter_fit(
                ax, df, "AO_mean", col,
                f"{label_lookup.get(col, row['label'])}: R={row['R_AO_vs_EP100']:.2f}, p={row['P']:.2g}",
                f"AO {df['AO_window'].iloc[0]}",
                f"EP100 {df['EP_window'].iloc[0]}",
                color_lookup.get(col, "tab:cyan"),
                annotate_members=False,
            )
        fig.suptitle("Top nominally significant asynchronous AO -> EPFlux relationships", fontsize=12)
        savefig(fig, f"{CASE}_AO_lead_EPFlux_top_significant_scatters")
        plt.show()

## 图：AO-EPFlux 任意 window/lag 密集扫描

**做什么**：在 `window >= 20 days`、`lag >= 0 days` 的限制下，对所有整数日 AO window、EPFlux window 和 lag 进行密集扫描，寻找最有潜力的 AO-leading-wave-activity 组合。

**怎么做**：AO 和 EPFlux 均先按成员对齐，然后用逐日累计和快速计算任意窗口平均。每个候选组合定义为：AO 从 `ao_start_doy` 开始平均 `window_days`，EPFlux 从 `ao_start_doy + delay_days` 开始平均同样长度。EPFlux 仍是 `EP100 = mean(-ep2)`，即 `100 hPa, 40-80N` 的垂直 EP-flux 分量，不是 divergence。

**多重检验提醒**：这个扫描会产生大量候选，因此单个最高相关很容易过拟合。输出里同时给 `BH-FDR q` 和局部邻域支持度 `local_support_same_sign`：后者统计同一 metric、同一符号、相近 start/window/delay 里是否也显著。更可信的是一片连续区域，而不是孤立最高点。

**科学问题**：如果 AO 对后续 wave activity 有潜在调制，最佳组合是否集中在物理上可解释的早冬/中冬窗口？是 all-wave、W1+W2、wave2 还是 rest 更稳定？

**运行后解读**：0008-01 的全局局部稳健最高组合集中在 5 月 `wave_rest`，例如 AO May04-May26 vs EP May06-May28，`R≈0.60, p≈4.5e-4`，但 `BH-FDR q≈1`，而且时间太晚，不适合作为早期 source 结论。若限制到 EP 窗口在 2 月底前结束，最有潜力的是早 1 月 AO 与稍晚 `wave2` 的负相关：AO Jan01-Jan21 vs EP Jan05-Jan25，`window=21 d, lag=4 d, R≈-0.789, p≈2.2e-7, q≈0.11`；相邻窗口给出类似结果，W1+W2 也为负、rest 为正。由于任意扫描检验数很大，尚无组合通过严格 `q<0.05`，所以这是探索性候选，不是最终显著结论。

In [ ]:
use_figure_dir("03_dense_arbitrary_window_lag_scan")


In [ ]:
# -----------------------------
# Dense arbitrary AO-window / EPFlux-lag scan
# -----------------------------
# Search all integer windows with window >= 20 days and non-negative lags.
# To reduce multiple-testing overinterpretation, keep all p<0.05 rows for BH-FDR
# and report a local-support score around the strongest candidates.
from scipy.stats import t as student_t

if ao is None:
    print("Skip dense arbitrary AO/EPFlux scan: projected AO is unavailable.")
else:
    DENSE_MIN_WINDOW_DAYS = 20
    DENSE_KEEP_P_LT = 0.05
    DENSE_KEEP_ABS_R_GE = 0.45
    DENSE_NEIGHBOR_DAYS = 3
    DENSE_MAX_TOP = 300

    def _values_by_member_short(da: xr.DataArray, common_members: Sequence[str]) -> np.ndarray:
        mids = [str(v) for v in da["member_short"].values]
        idx = [mids.index(m) for m in common_members]
        return da.isel(member=idx).values.astype(float)

    common_members = sorted(
        set(str(v) for v in ao["member_short"].values)
        & set(str(v) for v in ep_series["all_waves"]["member_short"].values)
    )
    print(f"Dense scan common members: {len(common_members)}")

    ao_values = _values_by_member_short(ao, common_members)
    ep_values = {wave: _values_by_member_short(ep_series[wave], common_members) for wave in WAVES}
    ep_values["wave1_plus_wave2"] = ep_values["wave1"] + ep_values["wave2"]

    ao_doy = date_to_doy(date_ao[: ao_values.shape[1]])
    ep_doy = date_to_doy(date_h[: ep_values["all_waves"].shape[1]])
    dense_max_doy = int(min(np.nanmax(ao_doy), np.nanmax(ep_doy)))
    dense_min_doy = int(max(np.nanmin(ao_doy), np.nanmin(ep_doy)))
    print(f"Dense scan DOY range: {dense_min_doy}-{dense_max_doy}")

    def _window_cumsums(values: np.ndarray, doys: np.ndarray, max_doy: int):
        sums = np.zeros((values.shape[0], max_doy + 1), dtype=float)
        counts = np.zeros((values.shape[0], max_doy + 1), dtype=float)
        for t_idx, doy in enumerate(np.asarray(doys, dtype=int)):
            if doy < 1 or doy > max_doy:
                continue
            col = values[:, t_idx]
            finite = np.isfinite(col)
            sums[:, doy] += np.where(finite, col, 0.0)
            counts[:, doy] += finite.astype(float)
        return np.cumsum(sums, axis=1), np.cumsum(counts, axis=1)

    def _segment_mean(csum: np.ndarray, ccnt: np.ndarray, start_doy: int, window_days: int) -> np.ndarray:
        end_doy = int(start_doy) + int(window_days) - 1
        total = csum[:, end_doy] - csum[:, start_doy - 1]
        count = ccnt[:, end_doy] - ccnt[:, start_doy - 1]
        out = np.full(total.shape, np.nan, dtype=float)
        np.divide(total, count, out=out, where=count > 0)
        return out

    def _corr_r_p_fast(x: np.ndarray, y: np.ndarray) -> tuple[float, float, int]:
        mask = np.isfinite(x) & np.isfinite(y)
        n = int(mask.sum())
        if n < 3:
            return np.nan, np.nan, n
        xv = x[mask]
        yv = y[mask]
        xv = xv - xv.mean()
        yv = yv - yv.mean()
        denom = math.sqrt(float(np.dot(xv, xv) * np.dot(yv, yv)))
        if denom == 0 or not np.isfinite(denom):
            return np.nan, np.nan, n
        r = float(np.dot(xv, yv) / denom)
        r = max(min(r, 0.999999999), -0.999999999)
        t_stat = abs(r) * math.sqrt((n - 2) / max(1e-15, 1.0 - r * r))
        p = float(2.0 * student_t.sf(t_stat, n - 2))
        return r, p, n

    ao_csum, ao_ccnt = _window_cumsums(ao_values, ao_doy, dense_max_doy)
    ep_cumsums = {key: _window_cumsums(val, ep_doy, dense_max_doy) for key, val in ep_values.items()}

    dense_metric_order = ["all_waves", "wave1", "wave2", "wave_rest", "wave1_plus_wave2"]
    dense_rows = []
    total_tests = 0
    latest_start = dense_max_doy - DENSE_MIN_WINDOW_DAYS + 1

    for window_days in range(DENSE_MIN_WINDOW_DAYS, dense_max_doy - dense_min_doy + 2):
        max_start = dense_max_doy - window_days + 1
        if max_start < dense_min_doy:
            continue
        ao_window_means = {
            start_doy: _segment_mean(ao_csum, ao_ccnt, start_doy, window_days)
            for start_doy in range(dense_min_doy, max_start + 1)
        }
        ep_window_means = {
            metric: {
                start_doy: _segment_mean(ep_cumsums[metric][0], ep_cumsums[metric][1], start_doy, window_days)
                for start_doy in range(dense_min_doy, max_start + 1)
            }
            for metric in dense_metric_order
        }
        for ao_start_doy, ao_vec in ao_window_means.items():
            for ep_start_doy in range(ao_start_doy, max_start + 1):
                delay_days = ep_start_doy - ao_start_doy
                for metric in dense_metric_order:
                    r, p, n = _corr_r_p_fast(ao_vec, ep_window_means[metric][ep_start_doy])
                    total_tests += 1
                    abs_r = abs(r) if np.isfinite(r) else np.nan
                    if np.isfinite(p) and (p < DENSE_KEEP_P_LT or abs_r >= DENSE_KEEP_ABS_R_GE):
                        dense_rows.append({
                            "metric": metric,
                            "label": WAVE_LABELS.get(metric, metric),
                            "R_AO_vs_EP100": r,
                            "P": p,
                            "abs_R": abs_r,
                            "ao_start_doy": ao_start_doy,
                            "ep_start_doy": ep_start_doy,
                            "window_days": window_days,
                            "delay_days": delay_days,
                            "AO_window": doy_window_label(ao_start_doy, window_days),
                            "EP_window": doy_window_label(ep_start_doy, window_days),
                            "n_members": n,
                        })

    dense_filtered = pd.DataFrame(dense_rows)
    print(f"Dense tests evaluated: {total_tests:,}; retained rows: {len(dense_filtered):,}")

    if dense_filtered.empty:
        print("No dense AO/EPFlux candidates passed the retention thresholds.")
    else:
        dense_filtered = dense_filtered.sort_values("P", kind="mergesort").reset_index(drop=True)
        ranks = np.arange(1, len(dense_filtered) + 1, dtype=float)
        q_raw = dense_filtered["P"].values * float(total_tests) / ranks
        q_monotone = np.minimum.accumulate(q_raw[::-1])[::-1]
        dense_filtered["bh_q"] = np.clip(q_monotone, 0, 1)
        dense_filtered["passes_nominal_filter"] = (dense_filtered["abs_R"] >= 0.5) & (dense_filtered["P"] < 0.05)
        dense_filtered["sign"] = np.sign(dense_filtered["R_AO_vs_EP100"]).astype(int)
        dense_filtered["ao_end_doy"] = dense_filtered["ao_start_doy"] + dense_filtered["window_days"] - 1
        dense_filtered["ep_end_doy"] = dense_filtered["ep_start_doy"] + dense_filtered["window_days"] - 1
        dense_filtered["early_source_window"] = dense_filtered["ep_end_doy"] <= mmdd_to_doy(2, 28)

        dense_sig = dense_filtered.loc[dense_filtered["passes_nominal_filter"]].copy()
        top_for_support = dense_sig.sort_values(["abs_R", "P"], ascending=[False, True]).head(DENSE_MAX_TOP).copy()
        support_values = []
        for _, row in top_for_support.iterrows():
            same = dense_sig.loc[
                (dense_sig["metric"] == row["metric"])
                & (dense_sig["sign"] == row["sign"])
                & ((dense_sig["ao_start_doy"] - row["ao_start_doy"]).abs() <= DENSE_NEIGHBOR_DAYS)
                & ((dense_sig["ep_start_doy"] - row["ep_start_doy"]).abs() <= DENSE_NEIGHBOR_DAYS)
                & ((dense_sig["window_days"] - row["window_days"]).abs() <= DENSE_NEIGHBOR_DAYS)
            ]
            support_values.append(int(len(same)))
        top_for_support["local_support_same_sign"] = support_values
        dense_promising = top_for_support.sort_values(
            ["local_support_same_sign", "abs_R", "bh_q", "P"],
            ascending=[False, False, True, True],
        ).reset_index(drop=True)

        dense_early = top_for_support.loc[top_for_support["early_source_window"]].sort_values(
            ["abs_R", "local_support_same_sign", "bh_q", "P"],
            ascending=[False, False, True, True],
        ).reset_index(drop=True)

        filtered_path = TABLE_DIR / f"{CASE}_AO_EPFlux_dense_arbitrary_window_lag_filtered.csv"
        top_path = TABLE_DIR / f"{CASE}_AO_EPFlux_dense_arbitrary_window_lag_top_promising.csv"
        early_path = TABLE_DIR / f"{CASE}_AO_EPFlux_dense_arbitrary_window_lag_early_source_top.csv"
        dense_filtered.to_csv(filtered_path, index=False)
        dense_promising.to_csv(top_path, index=False)
        dense_early.to_csv(early_path, index=False)
        print(f"Saved dense filtered candidates: {filtered_path}")
        print(f"Saved dense top promising candidates: {top_path}")
        print(f"Saved dense early-source candidates: {early_path}")
        print("\nTop dense candidates by local support, |R|, and BH q:")
        print(dense_promising.head(20)[[
            "metric", "R_AO_vs_EP100", "P", "bh_q", "local_support_same_sign",
            "AO_window", "EP_window", "window_days", "delay_days", "n_members"
        ]].to_string(index=False))
        print("\nTop dense early-source candidates by |R|, local support, and BH q (EP window ends by Feb28):")
        print(dense_early.head(20)[[
            "metric", "R_AO_vs_EP100", "P", "bh_q", "local_support_same_sign",
            "AO_window", "EP_window", "window_days", "delay_days", "n_members"
        ]].to_string(index=False))

        def _plot_dense_candidate(best_row: pd.Series, suffix: str, title_prefix: str):
            best_metric = str(best_row["metric"])
            best_window = int(best_row["window_days"])
            best_label = WAVE_LABELS.get(best_metric, best_metric)
            best_col = f"EP100_{best_metric}"
            best_df = ao_ep_async_table(
                ao, date_ao, ep_series, date_h,
                ao_start_doy=int(best_row["ao_start_doy"]),
                window_days=best_window,
                delay_days=int(best_row["delay_days"]),
            )

            max_start = dense_max_doy - best_window + 1
            heat_rows = []
            ao_means_best = {
                start_doy: _segment_mean(ao_csum, ao_ccnt, start_doy, best_window)
                for start_doy in range(dense_min_doy, max_start + 1)
            }
            ep_csum_best, ep_ccnt_best = ep_cumsums[best_metric]
            ep_means_best = {
                start_doy: _segment_mean(ep_csum_best, ep_ccnt_best, start_doy, best_window)
                for start_doy in range(dense_min_doy, max_start + 1)
            }
            for ao_start_doy, ao_vec in ao_means_best.items():
                for ep_start_doy in range(ao_start_doy, max_start + 1):
                    r, p, n = _corr_r_p_fast(ao_vec, ep_means_best[ep_start_doy])
                    heat_rows.append({
                        "ao_start_doy": ao_start_doy,
                        "delay_days": ep_start_doy - ao_start_doy,
                        "R": r,
                        "P": p,
                        "passes": abs(r) >= 0.5 and p < 0.05,
                    })
            heat_df = pd.DataFrame(heat_rows)
            pivot = heat_df.pivot(index="delay_days", columns="ao_start_doy", values="R").sort_index()

            color_lookup = {
                "all_waves": "black", "wave1": "tab:blue", "wave2": "tab:orange",
                "wave_rest": "tab:green", "wave1_plus_wave2": "tab:purple",
            }
            fig, axes = plt.subplots(1, 2, figsize=(14.5, 5.8), constrained_layout=True)
            scatter_fit(
                axes[0], best_df, "AO_mean", best_col,
                f"{title_prefix}: AO vs {best_label}",
                f"AO {best_df['AO_window'].iloc[0]}",
                f"EP100 {best_label} {best_df['EP_window'].iloc[0]}",
                color_lookup.get(best_metric, "tab:blue"),
                annotate_members=True,
            )
            im = axes[1].imshow(
                pivot.values,
                origin="lower",
                aspect="auto",
                cmap="RdBu_r",
                vmin=-1,
                vmax=1,
                extent=[pivot.columns.min(), pivot.columns.max(), pivot.index.min(), pivot.index.max()],
            )
            good = heat_df.loc[heat_df["passes"]]
            if not good.empty:
                axes[1].scatter(good["ao_start_doy"], good["delay_days"], s=18, facecolors="none", edgecolors="black", linewidths=0.8)
            xticks = [mmdd_to_doy(1, 1), mmdd_to_doy(2, 1), mmdd_to_doy(3, 1), mmdd_to_doy(4, 1), mmdd_to_doy(5, 1)]
            axes[1].set_xticks([t for t in xticks if pivot.columns.min() <= t <= pivot.columns.max()])
            axes[1].set_xticklabels([doy_window_label(int(t), 1)[:5] for t in axes[1].get_xticks()], rotation=30, ha="right")
            axes[1].set_xlabel("AO window start")
            axes[1].set_ylabel("EP lag after AO start (days)")
            axes[1].set_title(f"R heatmap: {best_label}, {best_window} d")
            fig.colorbar(im, ax=axes[1], shrink=0.9, label="R(AO, delayed EP100)")
            fig.suptitle(
                f"{title_prefix}; total tests={total_tests:,}\n"
                f"R={best_row['R_AO_vs_EP100']:.2f}, p={best_row['P']:.2g}, q={best_row['bh_q']:.2g}, support={int(best_row['local_support_same_sign'])}",
                fontsize=11,
            )
            savefig(fig, f"{CASE}_AO_EPFlux_dense_arbitrary_window_lag_{suffix}")
            plt.show()

        if not dense_promising.empty:
            _plot_dense_candidate(dense_promising.iloc[0], "best_overall_local_support", "Best overall by local support")
        if not dense_early.empty:
            _plot_dense_candidate(dense_early.iloc[0], "best_early_source", "Best early-source candidate")
